# 01 · Preparação dos dados

**Hipótese:** bairro mais violento → metro quadrado mais barato?

### Fontes

| Fonte | Conteúdo | Granularidade |
|---|---|---|
| `Crimes/SPDadosCriminais_{2022..2026}.xlsx` | Boletins de ocorrência da SSP-SP | 1 linha por ocorrência, estado inteiro |
| `Imoveis/GUIAS*.xlsx` | Guias de ITBI pagas | 1 linha por transação, só a capital |

### Localização das transações

A localização final é feita em `02_eda.ipynb`: usa o bairro informado no ITBI quando ele é reconhecido na base de crimes; para os demais registros, usa o bairro modal das transações com o mesmo CEP. Não há mais inferência do bairro por nome de rua.

Nesta execução, 38,4% das transações tinham bairro válido no ITBI e o CEP acrescentou 55,5 pontos percentuais, chegando a 94,0% de bairros resolvidos. A cobertura varia conforme os arquivos de entrada.

### Saídas

`crimes_sp.parquet` · `itbi_sp.parquet` · `setor_para_distrito.parquet` · `distritos_sp.geojson` · `setores_fiscais.geojson` — bases preparadas e geometria usada nas análises por distrito.

### Visão geral do pipeline

As bases são preparadas aqui e combinadas em `02_eda.ipynb`, seção 3:

```
Crimes (capital) ──► bairros válidos + centros medianos dos bairros
ITBI ─────────────► bairro informado, quando reconhecido
                  ► caso contrário, bairro modal por CEP
                  ▼
          itbi.bairro + itbi.lat/lon
```

A localização final usa somente o bairro informado no ITBI e a propagação por CEP. O dicionário de ruas e a relação setor fiscal → distrito deixam de participar desse merge.

In [1]:
import gzip
import io
import re
import urllib.request

import geopandas as gpd
import pandas as pd
from openpyxl import load_workbook

from comum import CACHE, RAIZ, contexto_https, em_unicos, sem_acento

CRIMES = RAIZ / "data" / "SP" / "Crimes"
IMOVEIS = RAIZ / "data" / "SP" / "Imoveis"
CACHE.mkdir(exist_ok=True)

sorted(p.name for p in CRIMES.glob("*.xlsx")) + sorted(p.name for p in IMOVEIS.glob("*.xlsx"))

['SPDadosCriminais_2022.xlsx',
 'SPDadosCriminais_2023.xlsx',
 'SPDadosCriminais_2024.xlsx',
 'SPDadosCriminais_2025.xlsx',
 'SPDadosCriminais_2026-3.xlsx',
 'GUIAS DE ITBI PAGAS (28012026) XLS.xlsx',
 'GUIAS DE ITBI PAGAS2025.xlsx',
 'GUIAS-DE-ITBI-PAGAS-2023.xlsx',
 'GUIAS-DE-ITBI-PAGAS-2024.xlsx',
 'GUIAS_DE_ITBI_PAGAS_(2020).xlsx',
 'GUIAS_DE_ITBI_PAGAS_(2021).xlsx',
 'GUIAS_DE_ITBI_PAGAS_12-2022.xlsx']

In [2]:
VAZIOS = {"", "NULL", "NAN", "NONE"}


def limpa_texto(valor):
    texto = sem_acento(valor)
    return None if texto in VAZIOS else texto


def cache(nome, construir):
    destino = CACHE / f"{nome}.parquet"
    if destino.exists():
        print(f"{nome}: lendo do cache")
        return pd.read_parquet(destino)
    tabela = construir()
    tabela.to_parquet(destino, index=False)
    print(f"{nome}: {len(tabela):,} linhas gravadas em {destino.name}")
    return tabela

## 1 · Crimes

Lido célula por célula (não `pd.read_excel` direto):

- **Estado inteiro.** Só ~40% das linhas são da capital — filtra na leitura, não depois.
- **Cabeçalho muda a cada ano.** `DESCR_PERIODO`/`DESC_PERIODO`, `CIRCUNCRIÇÃO`/`CIRCUNSCRIÇÃO`, `CIDADE` em 2022. Coluna localizada por prefixo; se nenhuma bater, `acha_coluna` falha com erro explícito.
- **Tipos misturados.** `LATITUDE` traz float, `0` e `'NULL'` na mesma coluna — limpa antes de salvar o parquet.

Geografia usada: *circunscrição* (onde o fato ocorreu). Coordenadas fora de SP (lat -24,0 a -23,3 · lon -46,9 a -46,3) viram nulas.

In [3]:
CAMPOS_CRIME = {
    "municipio": ("NOME_MUNICIPIO_CIRCUN",),
    "data_ocorrencia": ("DATA_OCORRENCIA_BO",),
    "periodo": ("DESC_PERIODO", "DESCR_PERIODO"),
    "local": ("DESCR_SUBTIPOLOCAL",),
    "bairro": ("BAIRRO",),
    "lat": ("LATITUDE",),
    "lon": ("LONGITUDE",),
    "natureza": ("NATUREZA_APURADA",),
    "delegacia": ("NOME_DELEGACIA_CIRCUN",),
}

JANELA_CIDADE = {"lat": (-24.0, -23.3), "lon": (-46.9, -46.3)}


def acha_coluna(cabecalho, prefixos, aba):
    posicao = next((i for i, c in enumerate(cabecalho) if c.startswith(prefixos)), None)
    if posicao is None:
        raise KeyError(f"{aba}: nenhuma coluna começa com {prefixos}")
    return posicao


def ler_aba_crime(caminho, aba):
    planilha = load_workbook(caminho, read_only=True)
    linhas = planilha[aba].iter_rows(values_only=True)
    cabecalho = [sem_acento(c) for c in next(linhas)]
    posicao = {
        novo: acha_coluna(cabecalho, prefixos, aba)
        for novo, prefixos in CAMPOS_CRIME.items()
    }
    coluna_municipio = posicao["municipio"]
    dados = [
        [linha[i] for i in posicao.values()]
        for linha in linhas
        if linha[coluna_municipio] == "S.PAULO"
    ]
    planilha.close()
    return pd.DataFrame(dados, columns=list(posicao)).drop(columns="municipio")


def limpa_crimes(tabela):
    for coluna in ["bairro", "natureza", "periodo", "local", "delegacia"]:
        tabela[coluna] = em_unicos(tabela[coluna], limpa_texto).astype("category")

    tabela["data_ocorrencia"] = pd.to_datetime(tabela["data_ocorrencia"], errors="coerce")
    tabela["ano"] = tabela["data_ocorrencia"].dt.year
    tabela["mes"] = tabela["data_ocorrencia"].dt.to_period("M").astype(str)

    for coluna, (minimo, maximo) in JANELA_CIDADE.items():
        valores = pd.to_numeric(tabela[coluna], errors="coerce")
        tabela[coluna] = valores.where(valores.between(minimo, maximo))

    return tabela[tabela["ano"].between(2022, 2026)].reset_index(drop=True)


def monta_crimes():
    partes = []
    for caminho in sorted(CRIMES.glob("SPDadosCriminais_*.xlsx")):
        planilha = load_workbook(caminho, read_only=True)
        abas = [a for a in planilha.sheetnames if re.fullmatch(r"(JAN-JUN|JUL-DEZ)_\d{4}", a)]
        planilha.close()
        for aba in abas:
            parte = ler_aba_crime(caminho, aba)
            print(f"  {aba}: {len(parte):,} ocorrências na capital")
            partes.append(parte)
    return limpa_crimes(pd.concat(partes, ignore_index=True))


crimes = cache("crimes_sp", monta_crimes)
crimes.shape

crimes_sp: lendo do cache


(2214258, 12)

In [4]:
crimes.notna().mean().mul(100).round(1).sort_values().to_frame("% preenchido")

,% preenchido
periodo,32.3
lat,81.0
lon,81.0
rua_key,91.9
local,99.5
bairro,99.7
data_ocorrencia,100.0
logradouro,100.0
natureza,100.0
delegacia,100.0


## 2 · Geometria da cidade (GeoSampa)

As malhas oficiais são usadas nas análises por distrito dos notebooks 04 e 05, não na localização de bairro do ITBI.

| camada | unidades | uso |
|---|---:|---|
| `distrito_municipal` | 96 | mapas e análise por distrito |
| `setor_fiscal` | 309 | vincular SQL do ITBI ao distrito |
| `setor → distrito` | — | distrito com maior sobreposição de área do setor |

In [5]:
WFS_GEOSAMPA = "https://wms.geosampa.prefeitura.sp.gov.br/geoserver/geoportal/wfs"
CRS_METRICO = 31983
TOLERANCIA_M = 20

CAMADAS = {
    "distritos_sp": ("distrito_municipal", ["cd_distrito_municipal", "nm_distrito_municipal"]),
    "setores_fiscais": ("setor_fiscal", ["cd_setor_fiscal"]),
}


def baixa_camada(camada, colunas):
    consulta = (
        f"{WFS_GEOSAMPA}?service=WFS&version=1.0.0&request=GetFeature"
        f"&typeName={camada}&outputFormat=application/json&srsName=EPSG:4326"
    )
    pedido = urllib.request.Request(
        consulta, headers={"Accept-Encoding": "identity", "User-Agent": "Mozilla/5.0"}
    )
    with urllib.request.urlopen(pedido, timeout=300, context=contexto_https()) as resposta:
        conteudo = resposta.read()
    if conteudo[:2] == b"\x1f\x8b":
        conteudo = gzip.decompress(conteudo)
    geometrias = gpd.read_file(io.BytesIO(conteudo))[colunas + ["geometry"]]
    geometrias["geometry"] = (
        geometrias.to_crs(CRS_METRICO).simplify(TOLERANCIA_M).to_crs(4326)
    )
    return geometrias


def cache_geo(nome):
    destino = CACHE / f"{nome}.geojson"
    if destino.exists():
        print(f"{nome}: lendo de {destino.name}")
        return gpd.read_file(destino)
    camada, colunas = CAMADAS[nome]
    geometrias = baixa_camada(camada, colunas)
    geometrias.to_file(destino, driver="GeoJSON")
    print(f"{nome}: {len(geometrias)} feições baixadas do GeoSampa ({destino.stat().st_size / 1024:,.0f} KB)")
    return geometrias


distritos = cache_geo("distritos_sp")
setores = cache_geo("setores_fiscais")
setores["setor_fiscal"] = setores["cd_setor_fiscal"].astype(str).str.zfill(3)
setores = setores.dissolve(by="setor_fiscal").reset_index()[["setor_fiscal", "geometry"]]

print(f"\n{len(distritos)} distritos · {len(setores)} setores fiscais")

distritos_sp: lendo de distritos_sp.geojson
setores_fiscais: lendo de setores_fiscais.geojson

96 distritos · 309 setores fiscais


In [6]:
def monta_setor_para_distrito():
    cruzamento = gpd.overlay(setores, distritos, how="intersection", keep_geom_type=True)
    cruzamento["area_m2"] = cruzamento.to_crs(CRS_METRICO).area
    return (
        cruzamento.sort_values("area_m2")
        .groupby("setor_fiscal")
        .agg(distrito=("nm_distrito_municipal", "last"), area_m2=("area_m2", "last"))
        .reset_index()
    )


setor_para_distrito = cache("setor_para_distrito", monta_setor_para_distrito)

print(f"\ndistritos alcançados: {setor_para_distrito['distrito'].nunique()} de {len(distritos)}")

setor_para_distrito: lendo do cache

distritos alcançados: 94 de 96


## 3 · ITBI

- **Sem cabeçalho no topo** — as planilhas são lidas sem cabeçalho, com os 28 nomes declarados; mantém apenas linhas cujo SQL da primeira coluna é numérico.
- `SQL` = setor (3 dígitos) + quadra (3) + lote (4) + dígito verificador.
- `setor_fiscal` são os três primeiros dígitos do SQL. É mantido para a análise por distrito nos notebooks 04 e 05; não participa da localização de bairro por CEP.

In [7]:
COLUNAS_ITBI = [
    "sql", "logradouro", "numero", "complemento", "bairro_itbi", "referencia", "cep",
    "natureza_transacao", "valor_transacao", "data_transacao", "vvr",
    "proporcao_transmitida", "vvr_proporcional", "base_calculo", "tipo_financiamento",
    "valor_financiado", "cartorio", "matricula", "situacao_sql", "area_terreno",
    "testada", "fracao_ideal", "area_construida", "uso_cod", "uso_desc", "padrao_cod",
    "padrao_desc", "acc",
]

NUMERICAS_ITBI = [
    "valor_transacao", "vvr", "vvr_proporcional", "base_calculo", "valor_financiado",
    "proporcao_transmitida", "area_terreno", "testada", "fracao_ideal",
    "area_construida", "acc", "uso_cod", "padrao_cod",
]


def limpa_itbi(tabela):
    tabela[NUMERICAS_ITBI] = tabela[NUMERICAS_ITBI].apply(pd.to_numeric, errors="coerce")
    tabela["sql"] = tabela["sql"].astype("int64").astype(str).str.zfill(11)
    tabela["setor_fiscal"] = tabela["sql"].str[:3]
    tabela["bairro_itbi"] = em_unicos(tabela["bairro_itbi"], limpa_texto)
    tabela["data_transacao"] = pd.to_datetime(tabela["data_transacao"], errors="coerce")
    tabela["ano"] = tabela["data_transacao"].dt.year
    tabela["mes"] = tabela["data_transacao"].dt.to_period("M").astype(str)

    for coluna in ["natureza_transacao", "uso_desc", "padrao_desc", "setor_fiscal", "situacao_sql"]:
        tabela[coluna] = em_unicos(tabela[coluna], limpa_texto).astype("category")

    descartaveis = ["complemento", "referencia", "cartorio", "tipo_financiamento", "matricula"]
    return tabela.drop(columns=descartaveis).reset_index(drop=True)


def monta_itbi():
    partes = []
    for caminho in sorted(IMOVEIS.glob("*.xlsx")):
        planilha = load_workbook(caminho, read_only=True)
        abas = [a for a in planilha.sheetnames if re.fullmatch(r"[A-Z]{3}-\d{4}", a)]
        for aba in abas:
            linhas = planilha[aba].iter_rows(values_only=True)
            dados = [
                (list(linha) + [None] * 28)[:28]
                for linha in linhas
                if isinstance(linha[0], (int, float))
            ]
            partes.append(pd.DataFrame(dados, columns=COLUNAS_ITBI))
        planilha.close()
        print(f"  {caminho.name}: {len(abas)} meses")
    return limpa_itbi(pd.concat(partes, ignore_index=True))


itbi = cache(
    "itbi_sp",
    lambda: monta_itbi().assign(numero=lambda tabela: tabela["numero"].astype("string")),
)
itbi.shape

itbi_sp: lendo do cache


(1302804, 27)

In [8]:
print(f"ITBI com setor fiscal presente na malha: {itbi['setor_fiscal'].astype(str).isin(set(setores['setor_fiscal'])).mean():.2%}\n")
itbi.notna().mean().mul(100).round(1).sort_values().to_frame("% preenchido")

ITBI com setor fiscal presente na malha: 100.00%



,% preenchido
bairro_itbi,62.1
acc,84.7
uso_desc,99.7
sql,100.0
rua_key,100.0
setor_fiscal,100.0
padrao_desc,100.0
padrao_cod,100.0
uso_cod,100.0
area_construida,100.0


In [9]:
pd.DataFrame(
    [
        ("crimes_sp", len(crimes), crimes["mes"].min(), crimes["mes"].max()),
        ("itbi_sp", len(itbi), itbi["mes"].min(), itbi["mes"].max()),
        ("setor_para_distrito", len(setor_para_distrito), "-", "-"),
    ],
    columns=["tabela", "linhas", "início", "fim"],
)

,tabela,linhas,início,fim
0,crimes_sp,2214258,2022-01,2026-08
1,itbi_sp,1302804,1994-07,NaT
2,setor_para_distrito,309,-,-


### Ressalvas

- A localização de bairro das transações usa o bairro reconhecido do ITBI e, quando ausente, o bairro modal por CEP. Não há inferência por nome de rua.
- A coordenada atribuída no notebook 02 é o centro mediano das ocorrências criminais do bairro; representa o bairro, não o endereço exato da transação.
- A relação `setor_fiscal → distrito` usa o distrito com maior sobreposição de área. Setores que cruzam divisas ficam associados ao distrito dominante.

➡️ Próximo: `02_eda.ipynb`.